# 2. Model configuration and runtime

Configuration is data; execution is code. Centralizing provider selection prevents every agent from reinventing API calls and makes mock/OpenRouter/Ollama switching explicit.

## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Load provider configuration, run the same runtime in mock or OpenRouter mode, and observe provider failures as events.

Architecture reference: [Day 5 diagrams D16](../../diagrams/source/day_05.md).

### Expected observation

Mock mode completes locally; configured OpenRouter uses the same runtime contract and records token usage. Exact IDs, timing, and live wording will vary.

## Concept briefing

## Configuration versus runtime

An agent configuration describes application-specific behavior: instructions, allowed
tools, model settings and limits. The runtime executes that configuration. A research
agent and a safe task agent should use one runtime without sharing inappropriate tools or
permissions.

A provider adapter hides API-specific request and response shapes behind a small
interface. Switching mock, OpenRouter or Ollama should not rewrite policy or the registry.
Provider metadata such as tokens, cost, latency and errors should still be preserved in
events.


In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

In [ ]:
import os
registry=build_demo_registry(); research=load_config("research_agent")
if os.getenv("OPENROUTER_API_KEY"):
    research.model.provider="openrouter"; research.model.model=os.getenv("OPENROUTER_MODEL","openai/gpt-oss-120b")
provider=build_provider(research.model); print("Provider:",research.model.provider)
runtime=HarnessRuntime(registry,provider)
result=runtime.run(research,"What is centralized by a harness?")
print(result.status,result.output)
for event in result.events: print(event)

## Live provider exercise

`build_provider` now supplies the tested OpenAI-compatible adapter. Change only `ModelConfig.provider`; registry, policy, and runtime remain unchanged. Ollama remains optional. Mock mode tests orchestration—it does not assess answer quality.

### Boundaries

Temperature and output limits belong in model configuration. Maximum tool steps belongs in agent/runtime configuration. API keys belong in environment variables, never JSON or notebooks.

## Your turn

Switch only the provider configuration, then compare event shapes rather than answer wording.

## Recap

Provider adapters isolate API differences from agent behavior. Name one responsibility that deliberately remains application-specific.